In [0]:
from pyspark.sql.functions import current_timestamp
schema_location = "/Volumes/arya_databricks_workspace/sanjay_autoloader/bronze_layer/schema/"
checkpoint_path = "/Volumes/arya_databricks_workspace/sanjay_autoloader/bronze_layer/checkpoints/"

raw_path = "/Volumes/arya_databricks_workspace/sanjay_autoloader/source"


df = (
spark.readStream
.format("cloudFiles")
.option("cloudFiles.format", "csv")
.option("cloudFiles.schemaLocation", schema_location)
.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
.option("cloudFiles.inferColumnTypes", "true")
.load(raw_path)
.withColumn("ingest_ts", current_timestamp())
)

In [0]:
(df.writeStream
.format("delta")
.option("checkpointLocation", checkpoint_path)
.option("mergeSchema", "true")
.trigger(availableNow=True)
# .trigger(processingTime="5 minutes")
.toTable("sanjay_autoloader.orders")

)

In [0]:
%sql
select * from sanjay_autoloader.orders